In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

ROOT = Path.cwd().resolve().parents[2]  # notebook lives in sw/tools/notebooks/
sys.path.insert(0, str(ROOT))
from sw.lib.pll_util import bin_phase_noise

VCO_DIR, PLL_DIR = ROOT / "results" / "vco", ROOT / "results" / "pll"
RESULT_CSV = None  # e.g. VCO_DIR / "vco_pnm_S5_20261021_100000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in VCO_DIR.glob("vco_pnm_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
CODES = ["vco_tune_coarse", "vco_current_min", "vco_current_max"]
KEY = ["f_vco_target", "set_div_freq"] + CODES  # one point
points = pd.read_csv(RESULT_CSV)
points["factor"] = points["factor"].fillna("")
points["pll_csv"] = points["pll_csv"].fillna("")
vco_traces = pd.read_csv(RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv"))
pll_traces = {name: pd.read_csv(PLL_DIR / (Path(name).stem + "_traces.csv")) for name in points["pll_csv"].unique()
              if name}
ok = points[points["status"] == "ok"]
print(f"{RESULT_CSV.name}: {len(points)} points, {len(ok)} measured")

INK, GRID = "#52514e", "#e4e3df"
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]
N_BINS = 300  # points per plotted curve (display only)
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz"], gridcolor=GRID,
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"), range=[np.log10(F_start), np.log10(F_stop)])


def setting(r):
    return f"c{r.vco_tune_coarse} min {r.vco_current_min} max {r.vco_current_max}, {r.f_vco_target / 1e6:.0f} MHz"


def of_point(t, r):
    """Rows of the trace table t belonging to point r."""
    sel = np.ones(len(t), dtype=bool)
    for c in KEY:
        sel &= t[c].to_numpy() == getattr(r, c)
    return t[sel]


def binned(df):
    """Curve df (freq, level_dbm) in N_BINS log-spaced bins over F_start-F_stop (linear power mean)."""
    df = df[(df["freq"] >= F_start) & (df["freq"] <= F_stop)].sort_values("freq")
    if len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(F_start, F_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def vco_curve(r):
    """Open-loop L(f) on the pad (Phase Noise mode) of point r."""
    t = of_point(vco_traces, r)
    return t[t["kind"] == "L_pnmode"]


def save(fig, name):
    fig.update_layout(template="plotly_white", font=dict(size=16))
    fig.write_html(RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}.html"))
    fig.show()


pd.DataFrame({
    "factor": points["factor"],
    "setting": [setting(r) for r in points.itertuples()],
    "PLL run": points["pll_csv"],
    "f_vco open loop [MHz]": points["f_vco_meas"] / 1e6,
    "Vctrl hint [V]": points["vctrl_hint"],
    "Vctrl read [V]": points["vctrl_read"],
    "carrier [dBm]": points["carrier_spectrum_dbm"],
    "jitter PLL [ps]": points["jitter_pll_ps"],
    "jitter open loop [ps]": points["time_jitter_pnm_ps"],
    "band open loop [Hz]": points["pnm_band"],
    "status": points["status"],
}).round(3)

vco_pnm_S5_20261021_143452.csv: 23 points, 13 measured


,factor,setting,PLL run,f_vco open loop [MHz],Vctrl hint [V],Vctrl read [V],carrier [dBm],jitter PLL [ps],jitter open loop [ps],band open loop [Hz],status
0,vco_current_max,"c9 min 9 max 0, 760 MHz",pll_test_current_max_S5_20261020_200224.csv,NaN,0.626,0.593,-0.752,300.466,NaN,NaN,error
1,vco_current_max,"c9 min 9 max 8, 760 MHz",pll_test_current_max_S5_20261020_200224.csv,759.881,0.618,0.586,-1.611,288.664,1751.560,1000-2e+07,ok
2,vco_current_max,"c9 min 9 max 12, 760 MHz",pll_test_current_max_S5_20261020_200224.csv,NaN,0.607,0.575,-1.346,310.343,NaN,NaN,error
3,vco_current_max,"c9 min 9 max 14, 760 MHz",pll_test_current_max_S5_20261020_200224.csv,NaN,0.589,0.559,0.432,322.589,NaN,NaN,error
4,vco_current_max,"c9 min 9 max 15, 760 MHz",pll_test_current_max_S5_20261020_200224.csv,759.582,0.536,0.517,-0.307,421.168,1824.071,1000-2e+07,ok
5,pll_1GHz,"c9 min 14 max 0, 1000 MHz",pll_test_current_max_S5_20261020_150031.csv,999.323,0.493,0.484,-0.937,34.385,2417.005,1000-2e+07,ok
6,pll_1GHz,"c9 min 12 max 2, 1000 MHz",pll_test_current_max_S5_20261020_150031.csv,998.936,0.491,0.482,-0.219,50.217,2441.609,1000-2e+07,ok
7,pll_1GHz,"c9 min 11 max 4, 1000 MHz",pll_test_current_max_S5_20261020_150031.csv,999.185,0.496,0.486,-0.463,64.840,2383.530,1000-2e+07,ok
8,pll_1GHz,"c9 min 9 max 6, 1000 MHz",pll_test_current_max_S5_20261020_150031.csv,999.227,0.492,0.482,-0.371,58.957,6131.298,1000-2e+07,ok
9,pll_1GHz,"c9 min 11 max 8, 1000 MHz",pll_test_current_max_S5_20261020_150031.csv,999.988,0.491,0.481,0.047,63.171,2320.117,1000-2e+07,ok


In [9]:
# PLL-matched points: phase noise referred to the VCO (pad + 20 log10 N), open loop (analyzer Phase Noise mode) and
# the PLL measurement at the same codes, divider and f_vco. One figure per point.
def pll_curve(r):
    """The PLL's full curve (L_trace) of point r: same run, codes, f_vco and the run's match (e.g. set_current=4)."""
    t = pll_traces[r.pll_csv]
    sel = (t["kind"] == "L_trace") & (t["f_vco"] == r.f_vco_target)
    for c in CODES:
        sel &= t[c] == getattr(r, c)
    for kv in str(r.pll_match).split() if isinstance(r.pll_match, str) else []:
        k, v = kv.split("=")
        sel &= t[k] == float(v)
    return t[sel]


for r in ok[ok["pll_csv"] != ""].itertuples():
    add_db = 20 * np.log10(r.total_div)
    fig = go.Figure()
    for name, t in (("open loop", vco_curve(r)), ("PLL", pll_curve(r))):
        t = binned(t)
        fig.add_scatter(x=t["freq"], y=t["level_dbm"] + add_db, mode="lines", name=name, line=dict(width=2),
                        hovertemplate=name + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Phase noise, {setting(r)}", height=550, legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS)
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    save(fig, f"c{r.vco_tune_coarse}_min{r.vco_current_min}_max{r.vco_current_max}_{r.f_vco_target / 1e6:.0f}MHz"
              "_vs_pll")

In [10]:
# One factor at a time: per factor (the points with that factor label, the other codes and f_vco fixed) one figure
# with the open-loop L(f) referred to the VCO, colour = the factor's value, and one with the RMS jitter of the Phase
# Noise mode (on the pad, over its band) against the value. vco_current_max: the 760 MHz PLL-matched points.
def factor_value(r, factor):
    return r.f_vco_target / 1e6 if factor == "f_vco" else getattr(r, factor)


FACTOR_TITLES = {"vco_current_max": "vco_current_max", "vco_current_min": "vco_current_min",
                 "vco_tune_coarse": "vco_tune_coarse (ring length)", "f_vco": "f_vco [MHz] (c9 band)"}
for factor, title in FACTOR_TITLES.items():
    sel = ok[[factor in f.split() for f in ok["factor"]]]
    if sel.empty:
        continue
    sel = sel.assign(value=[factor_value(r, factor) for r in sel.itertuples()]).sort_values("value")
    lo, hi = sel["value"].min(), sel["value"].max()

    fig = go.Figure()
    for r in sel.itertuples():
        t = binned(vco_curve(r))
        fig.add_scatter(x=t["freq"], y=t["level_dbm"] + 20 * np.log10(r.total_div), mode="lines",
                        name=f"{r.value:g}: {setting(r)}", line=dict(width=2, color=sample_colorscale(
                            "Viridis", [0.9 * (r.value - lo) / (hi - lo) if hi > lo else 0])[0]),
                        hovertemplate=f"{r.value:g}" + "<br>%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"Open-loop phase noise vs {title}", height=550, legend=dict(orientation="h", y=-0.2))
    fig.update_xaxes(title="offset from the carrier", **OFFSET_AXIS)
    fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
    save(fig, f"ofat_{factor}_phase_noise")

    fig = go.Figure()
    fig.add_scatter(x=sel["value"], y=sel["time_jitter_pnm_ps"], mode="lines+markers", line=dict(color=INK, width=2),
                    marker=dict(size=10), text=[setting(r) for r in sel.itertuples()],
                    hovertemplate="%{text}: %{y:.1f} ps<extra></extra>")
    fig.update_layout(title=f"Open-loop RMS jitter vs {title}", height=500)
    fig.update_xaxes(title=title, gridcolor=GRID)
    fig.update_yaxes(title="RMS jitter on the pad [ps]", gridcolor=GRID, rangemode="tozero")
    save(fig, f"ofat_{factor}_jitter")

In [11]:
# Averaged spectrum around the carrier per point and span (captured right after tuning): level relative to the
# carrier peak in the trace's RBW, against the offset from that peak. One figure per point and span.
spectra = vco_traces[vco_traces["kind"] == "spectrum"]
for r in points[points["status"].isin(["ok", "error"])].itertuples():
    for span, s in of_point(spectra, r).groupby("span"):
        peak = s.loc[s["level_dbm"].idxmax()]
        fig = go.Figure()
        fig.add_scatter(x=(s["freq"] - peak["freq"]) / 1e6, y=s["level_dbm"] - peak["level_dbm"], mode="lines",
                        line=dict(width=1.5), hovertemplate="%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
        fig.update_layout(title=f"Spectrum, {setting(r)} (span {span / 1e6:g} MHz, RBW {s['rbw'].iloc[0] / 1e3:g} kHz)",
                          height=550)
        fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
        fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
        save(fig, f"c{r.vco_tune_coarse}_min{r.vco_current_min}_max{r.vco_current_max}_{r.f_vco_target / 1e6:.0f}MHz"
                  f"_spectrum_{span / 1e6:g}MHz")